# Download UCF-Crime Videos

In [0]:
%pip install kaggle

In [0]:
import os
import pandas as pd

## 1. constants

In [0]:
CSV_PATH  = "/Volumes/workspace/default/ucf_crime/processed/spatiotemporal_ground_truth.csv"  # ground-truth boxes
VIDEO_DIR = "/Volumes/workspace/default/ucf_crime/videos"                                     # where videos are saved
DATASET   = "minhajuddinmeraj/anomalydetectiondatasetucf"                                     # Kaggle dataset ID (from its URL)
ORIGINALS_PREFIX = "Anomaly-Videos-Part"   # original videos; skips Augmented_* and duplicate class folders
 
SECRET_SCOPE = "kaggle"   # Databricks secret scope
SECRET_KEY   = "key"      # name the token was saved under
 
os.makedirs(VIDEO_DIR, exist_ok=True)

## 2. which videos are needed

In [0]:
boxes  = pd.read_csv(CSV_PATH)
train = pd.read_csv("/Volumes/workspace/default/ucf_crime/annotations_flat.csv")

wanted = set(boxes["video_id"].unique() + ".mp4") | set(train["video_name"].unique() + ".mp4")
have   = set(os.listdir(VIDEO_DIR))
 
todo = wanted - have
print(f"{len(wanted)} needed, {len(wanted & have)} already present, {len(todo)} to download")

##3. log into kaggle, token set before import

In [0]:
if todo:
    os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
 
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("Authenticated")
else:
    print("All videos present, nothing to do")

## 4. list dataset files

In [0]:
def list_all_files(dataset):
    names, token = [], None
    while True:
        res = api.dataset_list_files(dataset, page_token=token, page_size=200)
        names += [f.name for f in res.files]
        token = getattr(res, "next_page_token", None) or getattr(res, "nextPageToken", None)
        if not token:
            return names
 
if todo:
    files = list_all_files(DATASET)
    originals = [f for f in files if f.endswith(".mp4") and f.startswith(ORIGINALS_PREFIX)]
    print(f"{len(files)} files in dataset, {len(originals)} original videos")   # expect 950 originals

## 5. download missing videos

In [0]:
if todo:
    for f in originals:
        name = os.path.basename(f)
        if name in todo and name not in have:
            api.dataset_download_file(DATASET, f, path=VIDEO_DIR)
            have.add(name)

## 6. check result

In [0]:
vids    = os.listdir(VIDEO_DIR)
present = wanted & set(vids)
 
print(f"{len(present)} of {len(wanted)} present")
print("missing:", sorted(wanted - present))
print("non-mp4:", [v for v in vids if not v.endswith(".mp4")])
 
sizes_mb = {v: os.path.getsize(f"{VIDEO_DIR}/{v}") / 1e6 for v in present}
print("empty files:", [v for v, mb in sizes_mb.items() if mb == 0])
print(f"total size: {sum(sizes_mb.values()):.0f} MB")

## 7. unzip certain files

In [0]:
from collections import Counter

all_files = [os.path.join(r, f) for r, _, fs in os.walk(VIDEO_DIR) for f in fs]
print(len(all_files), "files total")
print(Counter(os.path.splitext(f)[1] for f in all_files))
print(all_files[:5])

In [0]:
import zipfile

zips = [f for f in all_files if f.endswith(".zip")]
for z in zips:
    with zipfile.ZipFile(z) as zf:
        zf.extractall(VIDEO_DIR)
    os.remove(z)
print(len(zips), "unzipped")